In [1]:
import pandas as pd
import numpy as np
df=pd.read_csv(r"C:\Users\Acer\Desktop\Data_Science_Projects\Feature Engineering\Creating Features\data\ames.csv")
df.head()

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YearSold,SaleType,SaleCondition,SalePrice
0,One_Story_1946_and_Newer_All_Styles,Residential_Low_Density,141.0,31770.0,Pave,No_Alley_Access,Slightly_Irregular,Lvl,AllPub,Corner,...,0.0,No_Pool,No_Fence,NaN,0.0,5,2010,WD,Normal,215000
1,One_Story_1946_and_Newer_All_Styles,Residential_High_Density,80.0,11622.0,Pave,No_Alley_Access,Regular,Lvl,AllPub,Inside,...,0.0,No_Pool,Minimum_Privacy,NaN,0.0,6,2010,WD,Normal,105000
2,One_Story_1946_and_Newer_All_Styles,Residential_Low_Density,81.0,14267.0,Pave,No_Alley_Access,Slightly_Irregular,Lvl,AllPub,Corner,...,0.0,No_Pool,No_Fence,Gar2,12500.0,6,2010,WD,Normal,172000
3,One_Story_1946_and_Newer_All_Styles,Residential_Low_Density,93.0,11160.0,Pave,No_Alley_Access,Regular,Lvl,AllPub,Corner,...,0.0,No_Pool,No_Fence,NaN,0.0,4,2010,WD,Normal,244000
4,Two_Story_1946_and_Newer,Residential_Low_Density,74.0,13830.0,Pave,No_Alley_Access,Slightly_Irregular,Lvl,AllPub,Inside,...,0.0,No_Pool,Minimum_Privacy,NaN,0.0,3,2010,WD,Normal,189900


In [2]:
#top features from Mutual info
top_features=[
    'OverallQual', 
    'Neighborhood',  
    'GrLivArea',     
    'YearBuilt',      
    'GarageArea ',     
    'TotalBsmtSF',    
    'GarageCars',      
    'BsmtQual'        
]

In [3]:
df['HouseAge']=2025-df['YearBuilt']

In [4]:
#log transform
df['Log_GrLivArea']=np.log1p(df["GrLivArea"])
df['Log_TotalBsmtSF']=np.log1p(df["TotalBsmtSF"])


In [5]:
#Garage Ratio Feature
df["GarageArea_per_Car"]=(df["GarageArea"]/df["GarageCars"].replace(0,1))

In [6]:
#Ordinal Encoding
qual_map={'Ex':5,'Gd':4,'TA':3,'Fa':2,'Po':1}
df['BsmtQual']=df["BsmtQual"].map(qual_map)
df['KitchenQual']=df["KitchenQual"].map(qual_map)

In [10]:
X=df.drop("SalePrice",axis=1)
y=df["SalePrice"]

In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

num_cols=X.select_dtypes(include=["int64","float64"]).columns
cat_cols=X.select_dtypes(include="object").columns

In [19]:
num_transformer=Pipeline(steps=[
    ("imputer",SimpleImputer(strategy="median"))
])
cat_transformer=Pipeline(steps=[
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("encoder",OneHotEncoder(handle_unknown="ignore"))
])

In [20]:
preprocessor=ColumnTransformer(
    transformers=[
        ("num",num_transformer,num_cols),
        ("cat",cat_transformer,cat_cols)
    ]
)

In [23]:
from sklearn.ensemble import RandomForestRegressor
pipeline=Pipeline(steps=[
    ("preprocess",preprocessor),
    ("model",RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

In [24]:
from sklearn.model_selection import cross_val_score

scores=cross_val_score(
    pipeline,
    X,
    y,
    scoring="neg_mean_absolute_error",
    cv=3
)
print("CV MAE : ", -scores.mean())

c:\Users\Acer\Desktop\Data_Science_Projects\venv\lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['BsmtQual' 'KitchenQual']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\Acer\Desktop\Data_Science_Projects\venv\lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['BsmtQual' 'KitchenQual']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\Acer\Desktop\Data_Science_Projects\venv\lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['BsmtQual' 'KitchenQual']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\Acer\Desktop\Data_Science_Projects\venv\lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [

CV MAE :  17040.2212475226
